---
# **Análise de Previsão de Demência**
---


🎯 **Objetivo específico:** Prever se o paciente apresenta sinais de demência.

🎯 **Objetivo geral:** Identificar as variáveis mais relevantes e propor uma análise baseada em modelos logísticos.

---


Desafio Estatística com Python - Regressão e Logística

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `Subject ID`: Identificador único do paciente
- `MRI ID`: Identificador único do exame
- `Group`: Classificação do paciente
- `Visit`: Identificador da visita de cada paciente
- `MR Delay`: Intervalo em dias entre os exames
- `M/F`: Gênero (M: masculino, F: feminino)
- `Hand`: Mão dominante
- `Age`: Idade do paciente (numérico)
- `EDUC`: Anos de escolaridade (numérico)
- `SES`: Status socioeconômico (1 a 5)
- `MMSE`: Escore do Mini Exame do Estado Mental (0 a 30)
- `CDR`: Clinical Dementia Rating (0 a 3)
- `eTIV`: Volume intracraniano estimado
- `nWBV`: Proporção de volume cerebral normalizado
- `ASF`: Fator de escala anatômica

Variável alvo categórica: **Group**

● Nondemented - será tratada para variável binária 0 

● Converted - será tratada para variável binária 1

● Demented - será tratada para variável binária 1

In [ ]:
# Importando as bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from IPython.display import display, Markdown

In [ ]:
# Configurações visuais dos gráficos

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2) 

In [ ]:
# Variáveis para reutilização

# data: dataframe contendo apenas as colunas de interesse
# nulos: colunas que possuem valores nulos
# var_features: seleção do df sem a variável alvo
# corr_rank: ranking de correlação com group
var_alvo = 'Group'

In [ ]:
# Lendo o dataframe

arquivo = 'oasis_longitudinal'
url = f'https://raw.githubusercontent.com/Squad-Nina-da-Hora/wmc-desafio-previsao-demencia/main/{arquivo}.csv'
df = pd.read_csv(url)

####  **Análise exploratória dos dados**

In [ ]:
# Conhecendo os dados

print(f'\nTotal de linhas: {df.shape[0]}')
print(f'Total de colunas: {df.shape[1]}')
print('-' * 50)

In [ ]:
# Verificando duplicatas

duplicados = df.duplicated().sum()
print(f'\nLinhas duplicadas na base: {duplicados}')

In [ ]:
# Verificando tipagem e nulos

info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df)) * 100,
    'Valores Únicos': df.nunique()
})
print('\n--- Diagnóstico de Tipagem e Qualidade ---\n')
display(info_df)

In [ ]:
# Armazenando colunas com valores nulos

nulos = df.columns[df.isna().any()].tolist()

print(f'Colunas com nulos identificadas: {nulos}')

In [ ]:
# Conhecendo os dados

df.head()

In [ ]:
# Conhecendo os dados

df.describe()

In [ ]:
# Conhecendo os dados

df.describe(include=['object', 'string'])

In [ ]:
# Conhecendo os dados

df['Group'].unique()

In [ ]:
# Conhecendo os dados

df['M/F'].unique()

In [ ]:
# Conhecendo os dados

df['Hand'].unique()

In [ ]:
# Selecionando apenas colunas de interesse

data = df.drop(['Subject ID', 'MRI ID', 'Visit', 'MR Delay', 'Hand'], axis=1)
data.head()

In [ ]:
# Variável categórica: % de group

data['Group'].value_counts(normalize=True)

In [ ]:
# Variável categórica: tratamento de group

data['Group'] = data['Group'].map({'Demented': 1, 'Converted': 1, 'Nondemented': 0})
data.head()

In [ ]:
# Variável categórica: % de gênero

data['M/F'].value_counts(normalize=True)

In [ ]:
# Variável categórica: tratamento de gênero

data['M/F'] = data['M/F'].map({'M': 1, 'F': 0})
data.head()

In [ ]:
# Fazendo a seleção do df sem a variável alvo (para reutilização)

var_features = [col for col in data.columns if col != var_alvo]
var_features

In [ ]:
# Calculando a matriz de correlação

corr = data.corr()
corr

In [ ]:
# Exibindo heatmap

plt.figure(figsize=(10,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', linewidth=0.5)
plt.title('Matriz de correlação', fontsize=12, fontweight='bold')
plt.show()

In [ ]:
# Analisando distribuição e desbalanceamento de Group

plt.figure(figsize=(8, 6))
ax = sns.countplot(data=data, x=var_alvo, palette=cores, hue=var_alvo, legend=False)
plt.title(f'Distribuição da Variável Alvo ({var_alvo})', fontsize=12, fontweight='bold')
plt.xlabel(f'{var_alvo}')
plt.ylabel('Contagem')

# Adicionando porcentagens nas barras
total = len(data)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{(height/total)*100:.1f}%', 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='bottom', xytext=(0, 3), 
                textcoords='offset points')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(var_features):
    sns.boxplot(data=data, x=var_alvo, y=var, ax=axes[i], palette=cores, hue=var_alvo, legend=False)
    axes[i].set_title(f'Boxplot: {var} por {var_alvo}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Histogramas

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
axes = axes.flatten()

for i, var in enumerate(var_features):
    sns.histplot(data=data, x=var, hue=var_alvo, kde=True, ax=axes[i], palette=cores, element='step')
    axes[i].set_title(f'Distribuição: {var}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Exibindo dispersões

sns.pairplot(data, hue=var_alvo, diag_kind='hist', palette=cores)

In [ ]:
# Ranqueando os principais preditores 

corr_rank = corr['Group'].abs().drop('Group').sort_values(ascending=False)
corr_rank

In [ ]:
display(Markdown(
        f'Com base na análise exploratória e no ranqueamento de correlação linear, os principais candidatos a preditores da variável alvo {var_alvo} são:\n'
        f'- {corr_rank.index[0]} (|r| = {corr_rank.iloc[0]:.2f}), \n'
        f'- {corr_rank.index[1]} (|r| = {corr_rank.iloc[1]:.2f}) e \n'
        f'- {corr_rank.index[2]} (|r| = {corr_rank.iloc[2]:.2f}).\n\n'
        f'Adicionalmente, as variáveis {corr_rank.index[3]} (|r| = {corr_rank.iloc[3]:.2f}) e {corr_rank.index[4]} (|r| = {corr_rank.iloc[4]:.2f}) demonstram associação intermediária, enquanto as demais características ({', '.join(corr_rank.index[5:])}) apresentam fraca correlação linear direta (|r| < 0.10) com o diagnóstico do grupo.'
))